# Шаг 6. Аналитика

Превращаем тысячи строк продаж в ответы на вопросы бизнеса. Функции из `app/analytics.py`.

In [ ]:
# Подготовка: чтобы ноутбук видел папку app и подхватывал изменения в коде без перезапуска
import sys

sys.path.append("..")

%load_ext autoreload
%autoreload 2

In [ ]:
import json

from app.analytics import (
    by_period,
    compare_periods,
    last_full_month,
    last_full_month_vs_previous,
    period_total,
)
from app.config import SYNTHETIC_DIR
from app.pipeline import prepare_events


def read_mapping(file_name: str) -> dict[str, str]:
    # Эталон разметки колонок из .json рядом с тестовым файлом
    return json.loads((SYNTHETIC_DIR / file_name).with_suffix(".json").read_text(encoding="utf-8"))


# Готовые события трёх бизнесов - одной командой каждый
shop, _ = prepare_events(SYNTHETIC_DIR / "shop.xlsx", read_mapping("shop.xlsx"))
cafe, _ = prepare_events(SYNTHETIC_DIR / "cafe.csv", read_mapping("cafe.csv"))
salon, _ = prepare_events(SYNTHETIC_DIR / "salon.xlsx", read_mapping("salon.xlsx"))
print(len(shop), len(cafe), len(salon))

## 1. by_period: итоги по периодам

In [ ]:
# Выручка магазина по месяцам: 2 года -> 24 числа вместо 3650 строк
monthly = by_period(shop, "месяц")
monthly.round(2)

In [ ]:
# Быстрый график, чтобы увидеть то, что заложено в данные:
# рост от года к году и сезонность (пик весной-летом, спад осенью-зимой)
monthly.plot(title="Выручка магазина по месяцам", marker="o", grid=True);

In [ ]:
# По дням - видны всплески в праздники (31 декабря, 8 марта...) и выходные
by_period(shop, "день").plot(title="Выручка магазина по дням", figsize=(12, 4), grid=True);

In [ ]:
# Не только деньги: количество проданных штук по неделям в кафе
by_period(cafe, "неделя", value="quantity").head()

In [ ]:
# Выручка всех трёх бизнесов по годам - одна и та же функция.
# ОСТОРОЖНО: годы неполные! В 2024 только 3 месяца (с октября), в 2026 - 9 (по сентябрь).
# Поэтому кажется, что 2026 хуже 2025, хотя на самом деле продажи растут.
# Сравнивать можно только одинаковые по длине периоды - этим займётся compare_periods
for name, events in [("магазин", shop), ("кафе", cafe), ("салон", salon)]:
    yearly = by_period(events, "год").round(2)
    print(name, dict(zip(yearly.index.year, yearly)))

## 2. compare_periods: сравнение двух периодов

In [ ]:
# Март 2026 против марта 2025: выручка магазина выросла на ~17%
compare_periods(shop, ("2025-03-01", "2025-03-31"), ("2026-03-01", "2026-03-31"))

In [ ]:
# Правильное сравнение: одинаковые 9 месяцев (январь-сентябрь) двух лет
compare_periods(shop, ("2025-01-01", "2025-09-30"), ("2026-01-01", "2026-09-30"))

In [ ]:
# Неправильное сравнение: весь 2025 против "всего" 2026, где данные только по сентябрь.
# Минус 11% - но это неправда. Функция предупреждает об этом
compare_periods(shop, ("2025-01-01", "2025-12-31"), ("2026-01-01", "2026-12-31"))

In [ ]:
# Последний день периода попадает целиком, даже с временем (у салона визиты с часами)
print("Салон, 31 марта:", period_total(salon, "2026-03-31", "2026-03-31"))
salon[salon["date"].dt.date.astype(str) == "2026-03-31"][["date", "object", "amount"]]

In [ ]:
# Ещё одна ловушка: январь против февраля в салоне - минус 15%.
# Но в феврале 28 дней, а в январе 31: на 3 дня работы меньше.
# Предупреждения нет - оба месяца внутри данных. Это нужно помнить, читая результат
compare_periods(salon, ("2026-01-01", "2026-01-31"), ("2026-02-01", "2026-02-28"))

## 3. last_full_month_vs_previous: как прошёл последний месяц

In [ ]:
# Как прошёл последний месяц в магазине? Ничего вводить не нужно - месяцы функция найдёт сама
last_full_month_vs_previous(shop)

In [ ]:
# Что если данные заканчиваются в середине месяца? Обрежем магазин по 15 сентября:
# сентябрь недописан, поэтому последний полный месяц - август
shop_until_15th = shop[shop["date"] <= "2026-09-15"]
print("Данные по:", shop_until_15th["date"].max().date())
print("Последний полный месяц:", last_full_month(shop_until_15th))

In [ ]:
# Ловушка с длиной месяцев решена: февраль (28 дней) -> март (31 день) в салоне.
# За месяц +28%, но часть этого роста - просто 3 лишних рабочих дня в марте.
# Честный рост - в среднем за день: около +16%
salon_until_march = salon[salon["date"] < "2026-04-01"]
last_full_month_vs_previous(salon_until_march)

### Попробуй сломать

In [ ]:
# У салона нет количества - понятная ошибка вместо столбца нулей
by_period(salon, "месяц", value="quantity")

In [ ]:
# Опечатка в периоде
by_period(shop, "квартал")

In [ ]:
# Бизнес только начал - данных меньше двух полных месяцев, сравнивать не с чем
last_full_month_vs_previous(shop[shop["date"] < "2024-11-20"])